# Week 3 - Data Wrangling

In this notebook we will practice data cleaning and wrangling using a *messy* version of the Titanic dataset.

Topics covered:
- Inspecting data
- Preprocessing
    - Categorical x Numerical Data
    - Fixing column types
    - Standardizing Categorical Values
    - Missing values (Identifying and Imputation)
- Feature Engineering
- Exporting and Loading Cleaned Data

### SETUP

In [ ]:
# Uncomment if needed.
# %pip install pandas numpy

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

def locate_data(filename):
    candidates = [Path(filename), Path('Week_3') / filename,
                  Path('raw') / filename, Path('../raw') / filename]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f'Upload {filename} to Colab or keep it in the supplied dataset folder.')

data_path = locate_data('messy_titanic.csv')

## Dataset

This is a modified Titanic teaching dataset with deliberately inconsistent formatting, artificial passenger identifiers and substituted names. It is not identical to the Week 2 CSV. The `USD` suffix is a formatting defect to remove, not evidence of a currency conversion.

| Column | Meaning |
| --- | --- |
| `passengerId` | Artificial passenger identifier |
| `survived` | Outcome: 0 = no, 1 = yes |
| `pclass` | Ticket class: 1, 2, 3 (ordered categories) |
| `name` | Substituted teaching name |
| `sex` | Sex label as supplied |
| `age` | Age in years; fractional ages are meaningful |
| `sibsp`, `parch` | Counts of siblings/spouses and parents/children aboard |
| `fare` | Recorded fare, sometimes containing an artificial text suffix |
| `adult_male` | Supplied Boolean attribute; not recomputed after imputation |
| `deck` | Deck letter; the input misspells this as `dreck` |
| `embark_town` | Embarkation town |

## 1. Inspecting the dataset
### Load the CSV once with the default separator. What goes wrong?

In [ ]:
# Your code and explanation here.

Inspect the first three text lines using `data_path.read_text().splitlines()[:3]`. This works on Windows and Colab. Identify the delimiter before trying again.

In [ ]:
# Your code and explanation here.

Read the file with the correct `sep` into `raw`. Preserve `raw` unchanged throughout the notebook.

In [ ]:
# Your code and explanation here.

### Q 1.1. Remove the saved index
Create `df` by dropping `Unnamed: 0` from `raw` and taking a copy. Explain why a DataFrame index is different from `passengerId`.

In [ ]:
# Your code and explanation here.

### Q 1.2. How many rows and columns are there?
Checkpoint after removing the saved index: 891 rows and 12 columns.

In [ ]:
# Your code and explanation here.

### Q.1.3 Check of each column data type

- Is every column data type the most appropriate for it?
- Are there any column with null/missing values?

In [ ]:
# Your code and explanation here.

### Q.1.4 Verify descritive statistics
- All numeric columns are being displayed?
- Those descritive statistcs are meaningfull for every column?

In [ ]:
# Your code and explanation here.

## 2. Preprocessing
### Q 2.1. Correct the column name
Rename `dreck` to `deck` in `df`. For example, `df = df.rename(columns={'old_name': 'new_name'})`.

In [ ]:
# Your code and explanation here.

<details>
<summary style="background-color: #fff3cd; padding: 10px; border-radius: 5px; cursor: pointer;">
    <h2 style="display: inline;">
        Understanding Numerical vs. Categorical Data
        <span style="color: red; font-size: 0.8em;">(click to expand/collapse)</span>
    </h2>
</summary>

It is important to understand **what type of data each column represents**.  
Choosing the **correct type** improves **memory usage**, **speeds up operations**, and makes **analysis more reliable**.

---

### **1. Numerical Data**
>
>These are values that represent quantities and support mathematical operations.
>
>**Examples:**
>- Age (`22`, `45.0`)
>- Fare (`71.283`, `15.50`)
>- Number of siblings (`3`, `0`)
>- Any measured or counted quantity
>
>**Two main subtypes:**
>- **int** → whole numbers (0, 1, 2…)
>- **float** → decimal numbers (0.0, 3.14…)
>
>**Numerical columns allow:**
>- statistics (mean, median, std)
>- plotting histograms
>- mathematical operations
>- machine learning models
---
### **2. Categorical Data**
>
>These represent **labels**, **groups**, or **categories**, not quantities.
>
>**Examples:**
>- Sex (`"male"`, `"female"`)
>- Embarkation port (`"C"`, `"Q"`, `"S"`)
>- Passenger class (`"First"`, `"Second"`, `"Third"`)
>- Deck (`"A"`, `"B"`, `"Unknown"`)
>
>**Types commonly used:**
>- **string (object)** → free text labels  
>- **category** → optimized categorical labels (recommended)
>
>**Categorical columns allow:**
>- grouping (`df.groupby("sex")`)
>- value counts
>- filtering
>- faster processing when stored as `category`
---
### **Why this distinction matters**
>
>If a column is stored with the wrong dtype:
>
>- A numeric column stored as string **cannot be computed**:  
>  `"45" + "10"` → `"4510"` (string concatenation!)
>- Numeric category codes need explicit interpretation; arithmetic on nominal codes is generally meaningless
>- Machine learning pipelines **fail or behave poorly**
>- Missing values become difficult to detect and fix
>- Memory usage increases (especially with text)
>
>**So the first step in Data Wrangling is:**
>
>👉 **Identify** whether each column is numerical or categorical  
>👉 **Fix** the dtype accordingly
</details>

## Fixing Column Types

Datasets often contain columns stored with the wrong data type.  
This usually happens when values are mixed (e.g., numbers + text), when the file was exported incorrectly, or when data is collected from multiple sources.

Working with incorrect types can break calculations, comparisons, plots, or machine-learning pipelines.  
So the first step is to **inspect** and **fix** them.

**Common strategies for fixing the column types are:**

___
>**Direct convertion:**
>
>If you know that the column will be able to be directly converted
>```python
>df['column_name'] = df['column_name'].astype(dtype)
>```
___
**1. Convert a column to numeric**
>
>Use when the column **should be a number**, but contains strings, symbols, or inconsistent formatting.
>
>```python
>df["column_name"] = pd.to_numeric(df["column_name"], errors="coerce")
>```
>`errors="coerce"` converts invalid values into **NaN**, which can later be imputed.
___

**2. Clean and convert a column that contains units or extra text**
>
>Useful for columns like "100 USD", "45 years", "200 km".
>
>```python
>df["column_name"] = (
>    df["column_name"]
>        .astype("string")
>        .str.replace(" USD", "", regex=False)
>)
>
># now casting to numeric dtype
>df["column_name"] = pd.to_numeric(df["column_name"], errors="coerce")
>```
___
**3. Convert a column to string (categorical text)**
>
>Use when the column should represent labels, not numbers.
>
>```python
>df["column_name"] = df["column_name"].astype("string")
>```
___
**4. Convert a column to category (optional optimization)**
>
>Useful for columns with repeated labels (e.g., "male", "female", "unknown").
>
>```python
>df["column_name"] = df["column_name"].astype("category")
>```
___
**5. Convert a column to datetime**
>
>For dates like "2020-01-05" or "05/01/2020".
>```python
>df["column_name"] = pd.to_datetime(df["column_name"], errors="coerce")
>```
___
**After fixing all types, it is good practice to check the result:**
>
>- `.info()`
>- `.dtypes`

### Q 2.2 Inspect each column, decide the most appropriate `dtype` and `fix` it:

For this file: remove the literal ` USD` suffix from fare before numeric conversion; distinguish original missingness from parse failures. Keep age as floating point to preserve fractional years. Use integer counts for `sibsp`/`parch`, categorical `pclass`, and validate 0/1 before converting `survived` to Boolean. Do not invent values for unknown outcomes.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

## Standardizing Categorical Values

Sometimes you may also find inconsistent or incorrect labels in categorical columns.  
For example, the same category may appear in multiple formats:

- "female", "Female", "F", "f"
- "male", "Male", "M", "m"

These inconsistencies make analysis, filtering, and grouping unreliable.  
To fix this, we **standardize** all categorical values into a consistent format.

---
**1. Convert everything to lowercase**
>
>This removes differences like "Female" vs "female".
>
>```python
>df["column_name"] = df["column_name"].astype("string").str.lower()
>```
---
**2. If there are many variations, a `mapping dictionary` might help:**
>
>```python
>mapping = {
>    "female": "female",
>    "f": "female",
>    "woman": "female",
>    "male": "male",
>    "m": "male",
>    "man": "male"
>}
>
>df["column_name"] = df["column_name"].replace(mapping)
>```
___
**Tips:**
>
>- Verify the frequencies of `categorical columns`
>```python
>df["column_name"].value_counts()
>```
>- Converting to `category` offers optimization in terms of memory footprint, comparinson speed and interpretability
>```python
>df["column_name"] = df["column_name"].astype("category")
>```

### Q.2.3 Verify if any categorical column needs to be standardized
- Inspect columns of dtype `category`, `object`, `bool`, `int`
- Inspect using `df['column'].value_counts()`

Normalize `sex` and `embark_town` with `str.strip().str.lower()`. Use `string` during cleaning to preserve missingness. Check observed labels before converting to `category`; do not silently turn unexpected labels into missing values.

In [ ]:
# Your code and explanation here.

## Missing values

Most real-world datasets contain missing or inconsistent values.

We must decide how to handle them.

**The main strategies are:**
___
**1. Remove rows with missing values**

>Useful when only a few rows contain missing values.
>
>- Removing rows can change the population represented, even if few rows are removed. Justify the decision.
>
>```python
>df_clean = df.dropna()
>```
___
**2. Remove columns with too many missing values**

>Useful when a column is mostly empty (e.g., >80% missing maybe even lower threshold)
>
>- The column is not essential for the analysis
>
>```python
>df_clean = df.drop(columns=["column_name"])
>```
___
**3. Impute missing values (numeric columns)**

>Common inputs:
>
>- Mean
>
>- Median (more robust to outliers)
>
>- Constant value
>
>```python
># mean
>df["column_name"] = df["column_name"].fillna(df["column_name"].mean())
>
># median
>df["column_name"] = df["column_name"].fillna(df["column_name"].median())
>
># constant value
>df["column_name"] = df["column_name"].fillna(1)
>```
___
**4. Impute missing values (categorical columns)**

>Typical strategies:
>
>Replace with the most frequent category (mode)
>
>Replace with a special label such as "Unknown"
>
>
>```python
># mode
>df["column_name"] = df["column_name"].fillna(df["column_name"].mode()[0])
>
># special label
>df["column_name"] = df["column_name"].fillna("Unknown")
>```
___
**There are more advanced methods of imputation but those above cover most of the problems that you might have initially.**

### Q.2.4 Check for missing values in every column:
- Try to obtain a sum of missing values in each column

In [ ]:
# Your code and explanation here.

### Q.2.5 Which column is the most problematic in terms of missing values?

In [ ]:
# Your code and explanation here.

### Q.2.6 Let's find out the percentage of missing values in the most problematic column.

In [ ]:
# Your code and explanation here.

### Q.2.7 Define a function that will check for missing values in percentage each column that you provide as a param.

**Example:**

```python
def percentage_missing_values(df: pd.DataFrame, col: str) -> float:

    # your code here
    
    return percentage_of_missing
```


In [ ]:
# Your code and explanation here.

### Q 2.8. Apply an explicit missing-value policy
For this descriptive exercise, preserve every passenger and create `age_imputed` and `fare_imputed` flags before filling age and fare with their respective medians. Fail clearly if a median cannot be estimated. Keep fractional ages. Fill missing `deck` and `embark_town` with `unknown`, then convert these columns to `category`.

A median is an estimate, not the original value. High missingness alone does not prove a column is useless. Compare group sizes before adopting group-specific medians.

Checkpoint before filling: age 177 missing, fare 45, deck 688, embarkation town 2.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

In [ ]:
# Your code and explanation here.

## 3. Feature Engineering

Now that the dataset is clean, with corrected data types, standardized categorical values, and missing values imputed, we can begin improving the dataset by **creating new features**.

Feature Engineering helps models extract more meaningful patterns by transforming the raw data into something more informative and more useful for analysis or machine learning.

---
>**1.Mathematical combinations**
>Create new features by combining existing numeric features.
>
>**Example: family size**
>```python
>df["family_size"] = df["sibsp"] + df["parch"] + 1
>```
---
>**2. Binarizing or grouping categories**
>Transform categories into simpler or grouped labels.
>
>**Example: traveling alone vs not**
>```python
>df["is_alone"] = (df["family_size"] == 1)
>```
---
>**3. Extracting information from existing columns**
>
>**Example: title extraction for a different dataset with names such as `Smith, Mr. John`**
>
>This file uses teaching names without titles; do not apply this pattern to it.
>```python
>df["title"] = df["name"].str.extract(r",\s*([^\.]*)\.", expand=False)
>df["title"] = df["title"].str.strip().str.lower()
>```
---
>**4. Discretization (binning)**
>Convert continuous variables into intervals.
>
>**Example using `cut`:**
>```python
>df["age_group"] = pd.cut(
>    df["age"],
>    bins=[0, 12, 18, 35, 60, 100],
>    labels=["child", "teen", "young_adult", "adult", "senior"]
>)
>```
>**-  `pd.cut` will consider intervals as `( ]`**
>**- `open on left (exclusive)` and `closed on right (inclusive)`**
>
>**Example using `qcut` (data will be split in equal parts)**
>```python
>df["age_group"] = pd.qcut(
>    df["age"],
>    q=5,
>    labels=["child", "teen", "young_adult", "adult", "senior"]
>)
>```
---
>**5. Encoding boolean or numeric signals**
>
>**Example: indicator for high fare**
>```python
>df["high_fare"] = df["fare"] > df["fare"].median()
>```
---
>**6. Interaction features**
>Features created by combining multiple existing ones.
>
>**Example: fare divided by recorded family party size**
>
>This ratio is a constructed feature, not a verified per-person ticket price.
>```python
>df["fare_per_person"] = df["fare"] / df["family_size"]
>```
---
**Example Creating feature set**
>
>```python
>df["family_size"] = df["sibsp"] + df["parch"] + 1
>df["is_alone"] = (df["family_size"] == 1)
>
>df["title"] = df["name"].str.extract(r",\s*([^\.]*)\.", expand=False)
>df["title"] = df["title"].str.strip().str.lower()
>
>df["age_group"] = pd.cut(
>    df["age"],
>    bins=[0, 12, 18, 35, 60, 100],
>    labels=["child", "teen", "young_adult", "adult", "senior"]
>)
>
>df["high_fare"] = df["fare"] > df["fare"].median()
>df["fare_per_person"] = df["fare"] / df["family_size"]
>
>df.head()
>```
---
### Q 3.1. Create a new column called `is_child` that is `True` when **age < 12**.

In [ ]:
# Your code and explanation here.

### **Q 3.2.** Create a column `family_size` based on `sibsp` and `parch`.

In [ ]:
# Your code and explanation here.

### Q 3.3. Create a `class` column based on the `Ticket Class`.
- **1 -> First**
- **2 -> Second**
- **3 -> Third**

- **Set the `dtype` of this column to `category`.**

In [ ]:
# Your code and explanation here.

### Q.3.4. Create a `embarked` column based on the Port where the passenger embarked.
- **Use the initials as the category value.**
- **Set the `dtype` of this column to `category`.**
Map Southampton/Cherbourg/Queenstown to S/C/Q after normalization; preserve unknown as `unknown`.

In [ ]:
# Your code and explanation here.

### Q 3.5. Create a `fare_bin` column by splitting fare into 4 quantiles using `pd.qcut`.
- use `labels` as a list of `'Low', 'Medium', 'High', 'Very High'`
Quantile boundaries can coincide when values repeat; inspect boundaries and do not assume four distinct bins exist in every dataset.

In [ ]:
# Your code and explanation here.

### Q 3.6. Create a column `age_fare_ratio` equal to `age / fare`, and comment on whether this feature makes sense.
- be careful about zero-division
Represent ratios with zero fare as missing; explain the units and why this feature may have little practical meaning.

In [ ]:
# Your code and explanation here.

### Q 3.7. Create `age_group` **(you decide the bins and labels)** and count how many passengers exist in each `age_group`.

In [ ]:
# Your code and explanation here.

### Q 3.8. Extract a surname
For these substituted teaching names, take the last whitespace-separated token. Explain why this is only a dataset-specific convention and not a general rule for human names.

In [ ]:
# Your code and explanation here.

### **Q 3.9.** Using the engineered features, compute a summary showing:
- average `fare` per `age_group`
- average `family_size` per `class`

In [ ]:
# Your code and explanation here.

## 4. Export Dataset

Now that our dataset is clean, we might want to save time by not needing to preprocess it again.

Exporting it is a good way to avoid running all cleaning steps every time you start your notebook.

There are many formats you can choose, and the most common are:

---

### ✔️ Export to CSV
>```python
>df_clean.to_csv("titanic_clean.csv", index=False)
>```
>
>- `index=False` avoids saving the DataFrame index as an extra column.
>- CSV files are human-readable and compatible with most tools.
---
### ✔️ Export to Excel
>```python
>df_clean.to_excel("titanic_clean.xlsx", index=False)
>```
>
>- Requires `openpyxl` or `xlsxwriter` installed.
>- Useful when sharing data with non-programmers.
---
### ✔️ Export to JSON
>```python
>df_clean.to_json("titanic_clean.json", orient="records")
>```
>
>**Common orientations:**
>
>- `"records"` → list of dictionaries (most common)
>- `"split"` → dictionary with `index`, `columns`, `data`
>- `"table"` → includes metadata, useful for interoperability
---
### ✔️ Loading Back the Dataset Later
>```python
>df_loaded = pd.read_csv("titanic_clean.csv")
>```
>
>**or:**
>
>```python
>df_loaded = pd.read_json("titanic_clean.json")
>```
---
**if you are curious, search for `pickle` and `parquet`.**

Keep the preparation code and source data so results remain reproducible. CSV does not preserve categorical dtypes; restore them explicitly when needed.

### Q 4.1. Export the dataset to `csv`. Set `index=True` as an additional param.

Write all exercise exports under `outputs/`; never overwrite the supplied CSV.

In [ ]:
# Your code and explanation here.

### Q 4.2 Load into a variable named `df_loaded`

In [ ]:
# Your code and explanation here.

**You also could just drop the artificial index column...** `df.drop(columns=['col'], inplace=True)`

### Q 4.3 Export the dataset to `csv`. Set `index=False` as an additional param.

In [ ]:
# Your code and explanation here.

## 5. Final Clean Dataset

### Load the Clean Dataset

- **Test first with a different variable name to avoid overwriting by accident.**
Load `outputs/titanic_clean.csv` into a new variable. Verify row count, identifiers, column names, numeric values and the absence of an artificial index column. Confirm `raw` still matches the source file. The undefined zero-fare ratio may remain missing by design.

In [ ]:
# Your code and explanation here.

## 6. Categorical encoding and scaling

After cleaning the values, we can choose how to **represent categories** and **scale numerical features**.

---
**1. Categorical dtype vs. numerical encoding**

> `category` is a Pandas dtype. It does not automatically create numerical input features.
>
> For nominal labels, arbitrary numbers introduce an artificial order. `pclass` has a natural order, but equal numerical distances still need justification.

---
**2. One-hot encoding**

> Create one indicator column for each category: **1** means the row belongs to that category, and **0** means it does not.
>
> **Example: communication channels**
>
> ```python
> channels = pd.Series(['email', 'web', 'email'])
> channel_features = pd.get_dummies(
>     channels, prefix='channel', dtype=int
> )
> channel_features
> ```
>
> **Check:** each row should have exactly one active indicator when each observation has one known category.
>
> ```python
> channel_features.sum(axis=1)
> ```
>
> Decide explicitly how missing or unknown categories are represented. Different observed categories can produce different columns.

---
**3. Standardizing numerical values**

> Z-score standardization subtracts the mean and divides by the standard deviation:
>
> **`z = (x - mean) / standard_deviation`**
>
> **Example: measured durations**
>
> ```python
> durations = pd.Series([10.0, 20.0, 30.0])
> duration_mean = durations.mean()
> duration_std = durations.std(ddof=0)
> duration_zscore = (durations - duration_mean) / duration_std
> duration_zscore
> ```
>
> Use **`ddof=0`** for the population standard deviation in this lesson. The transformed values have mean approximately zero and standard deviation one.

---
**4. Handling a constant column**

> A constant column has standard deviation **zero**. Replace that denominator with 1: after centering, every transformed value is zero.
>
> ```python
> constant_values = pd.Series([5.0, 5.0, 5.0])
> constant_std = constant_values.std(ddof=0)
> safe_std = 1.0 if constant_std == 0 else constant_std
> constant_zscore = (
>     constant_values - constant_values.mean()
> ) / safe_std
> ```
>
> **Remember:** scaling changes units. It does not remove outliers or make a distribution normal. The full-table statistics here serve a descriptive exercise.

### Q 6.1. One-hot encoding

Encode `embark_town` (including `unknown`) with prefix `port` and integer indicators. Verify one active indicator per row. Explain why separate datasets might produce different columns.

In [ ]:
# Your code and explanation here.

### Q 6.2. Safe scaling

Copy `age` and `fare` into `numeric_features`, adding a constant column equal to 5. Calculate means and population standard deviations, replace zero denominators with 1 and standardize. Check means near zero, standard deviations near one for nonconstant columns and zeros for the constant column.

In [ ]:
# Your code and explanation here.

### Q 6.3. Assemble a feature table

Create `is_alone = family_size == 1` using the feature from Q 3.2. Combine scaled age/fare, the port indicators, `family_size` and `is_alone` into `features`. Keep passenger IDs separately as the table index for alignment. Exclude `survived` (the future target), names, ID values as inputs, and the demonstration constant. Verify alignment and finite values.

In [ ]:
# Your code and explanation here.

## 7. Checks and export

Before saving the result, verify that the transformations produced what you intended. Keep the source data and preparation code so the work can be reproduced.

---
**1. Check the prepared data**

> Use **`assert`** to state conditions that must hold. A failed assertion tells you to investigate before exporting.
>
> **Example: a small prepared table**
>
> ```python
> prepared = pd.DataFrame({
>     'record_id': ['A', 'B'],
>     'duration': [10.0, 20.0]
> })
> assert prepared['record_id'].is_unique
> assert prepared['duration'].notna().all()
> assert prepared['duration'].ge(0).all()
> ```
>
> Choose checks that match the task. Some fields may legitimately remain missing.

---
**2. Save and read back**

> Save without an extra DataFrame index, then load the file into a **different variable**.
>
> ```python
> Path('outputs').mkdir(exist_ok=True)
> prepared.to_csv('outputs/prepared_example.csv', index=False)
> reloaded = pd.read_csv('outputs/prepared_example.csv')
>
> assert reloaded.shape == prepared.shape
> assert reloaded.columns.tolist() == prepared.columns.tolist()
> ```
>
> **Remember:** CSV does not automatically preserve categorical dtypes. Check identifiers and numerical values too; matching shape alone does not guarantee matching contents.

### Q 7.1. Validate and reload

Check row count and unique IDs, nonnegative ages/fares, allowed categories and missingness in required columns. Export the clean table and the feature table under `outputs/`, with the identifier as an explicit column and without a saved DataFrame index. Reload and check identifiers, columns and numeric values. Confirm that `raw` is unchanged.

In [ ]:
# Your code and explanation here.

## 8. Cleaning and validating dates

A registration file may combine an online form, a spreadsheet and manual entries. Even when the field means the same thing, its text representation may differ. This synthetic example reproduces common data-entry problems without changing the Titanic table.

---
**1. Inspect the original values before converting**

> **Examples from different input systems:**
>
> | Original value | What needs attention? |
> |---|---|
> | `2026-09-15` | ISO date from an online form |
> | ` 2026-09-15 ` | Extra spaces around a valid date |
> | `15/09/2026` | Day/month/year from a spreadsheet |
> | `03/09/2026` | Ambiguous without a documented day/month convention |
> | `2026-09-30 23:59:00` | Timestamp at the end of the permitted month |
> | `N/A`, `NULL`, blank text | Missing-value markers |
> | `31/09/2026` | Impossible calendar date |
> | `2026-09-1O` | Letter O typed instead of digit zero |
> | `pending` | Status text entered in the date field |
>
> Do not silently repair uncertain values. A plausible correction can still be wrong.

---
**2. Normalize spaces and documented missing markers**

> Keep the source unchanged and work on a separate Series. The nullable `string` dtype preserves actual missing values.
>
> ```python
> text = source_dates.astype('string').str.strip()
> source_missing = source_dates.isna()
> missing_marker = text.str.lower().isin(['', 'n/a', 'null'])
> normalized = text.mask(missing_marker)
> ```
>
> Here, these markers are explicitly documented as missing. `pending` is **invalid text**, not a missing marker. Record source missingness separately from missing markers.

---
**3. Parse only documented formats**

> This file accepts **ISO dates**, **day/month/year dates**, and either format followed by a local time in **HH:MM:SS**. Slash dates always use **day/month/year**: `03/09/2026` means **3 September**.
>
> ```python
> parsed = pd.to_datetime(
>     normalized, format='%Y-%m-%d', errors='coerce'
> )
> slash_dates = pd.to_datetime(
>     normalized, format='%d/%m/%Y', errors='coerce'
> )
> parsed = parsed.fillna(slash_dates)
> ```
>
> The worked example below additionally checks text shapes and handles timestamps. **`errors='coerce'`** turns conversion failures into **`NaT`**. It does not fix impossible dates.
>
> Without a documented convention, an ambiguous value such as `03/09/2026` needs clarification. Do not guess the format from individual values. Month/day/year and timezone-bearing timestamps are not accepted in this exercise.

---
**4. Validate the period without losing the final day's times**

> All timestamps in this example represent the same local time convention. Retain the time and use a half-open interval:
>
> ```python
> in_september = (
>     parsed.ge('2026-09-01') & parsed.lt('2026-10-01')
> )
> ```
>
> This accepts **30 September at 23:59:00** and rejects **1 October at 00:00:00**. An upper bound of `2026-09-30` would mean midnight at the start of the final day.
>
> A missing or impossible date becomes `NaT`; an out-of-period date can parse successfully. These are different problems.

---
**5. Preserve the source and explain decisions**

> Keep the original date text alongside the parsed value. Distinguish **missing**, **invalid** and **outside-period** values. Do not invent dates or replace uncertain entries with today's date.

---
**Worked example: compare source, normalized text and parsed timestamps**

> Run the following example. Notice which values can be repaired safely, which remain invalid, and which parse successfully but fall outside September.

In [ ]:
date_example = pd.Series([
    '2026-09-01', ' 2026-09-15 ', '15/09/2026', '03/09/2026',
    '2026-09-30 23:59:00', '30/09/2026 18:45:00',
    None, '', '   ', 'N/A', 'NULL',
    '31/09/2026', '2026-09-1O', 'pending', '09/15/2026',
    '2026-08-31 23:59:59', '2026-10-01 00:00:00', '2025-09-15'
])
example_text = date_example.astype('string').str.strip()
example_text = example_text.mask(example_text.str.lower().isin(['', 'n/a', 'null']))
date_formats = {
    r'\d{4}-\d{2}-\d{2}': '%Y-%m-%d',
    r'\d{2}/\d{2}/\d{4}': '%d/%m/%Y',
    r'\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}': '%Y-%m-%d %H:%M:%S',
    r'\d{2}/\d{2}/\d{4} \d{2}:\d{2}:\d{2}': '%d/%m/%Y %H:%M:%S'
}
parsed_example = pd.Series(pd.NaT, index=date_example.index, dtype='datetime64[ns]')
for pattern, date_format in date_formats.items():
    matching = example_text.str.fullmatch(pattern, na=False)
    parsed_example.loc[matching] = pd.to_datetime(
        example_text.loc[matching], format=date_format, errors='coerce'
    )
example_in_period = parsed_example.ge('2026-09-01') & parsed_example.lt('2026-10-01')
print(pd.DataFrame({
    'original': date_example, 'normalized': example_text,
    'parsed': parsed_example, 'in_september': example_in_period
}).to_string(index=False))
assert parsed_example.iloc[3] == pd.Timestamp('2026-09-03')
assert example_in_period.iloc[4]  # Final-day evening is accepted.
assert not example_in_period.iloc[16]  # October midnight is rejected.

### Practice dataset

> **36 synthetic workshop registrations** combine web-form dates, day/month/year spreadsheet entries and local timestamps, including recoverable formatting problems and records requiring rejection.
>
> Accept registrations from **September 2026**. Use only the **four documented formats** from the worked example; all slash dates are day/month/year. Trim spaces and treat blank text, `N/A` and `NULL` as missing. Do not guess dates from typos, accept US month/day order or impute missing dates.
>
> Preserve **`date_raw`** and the original text. These teaching rules extend the date examples; the service-ticket mini-project has its own requirements in the brief.

In [ ]:
date_raw = pd.DataFrame({
    'registration_id': [f'R{i:03d}' for i in range(1, 37)],
    'registered_on': [
        '2026-09-01',
        '2026-09-30',
        ' 2026-09-05 ',
        '2026-09-12 09:30:00',
        '15/09/2026',
        '03/09/2026',
        '30/09/2026 23:59:00',
        '2026-09-01 00:00:00',
        '2026-09-30 23:59:59',
        ' 21/09/2026 ',
        '07/09/2026 08:15:00',
        '2026-09-22',
        '2026-09-23 14:05:10',
        '24/09/2026',
        '25/09/2026 17:30:00',
        '2026-09-26',
        '2026-09-27',
        '28/09/2026',
        None,
        None,
        '',
        '   ',
        'N/A',
        'NULL',
        '2026-02-30',
        '31/09/2026',
        '2026-13-01',
        '2026-09-1O',
        'pending',
        '09/15/2026',
        '2026-09-15 25:10:00',
        '2026-08-31 23:59:59',
        '2026-10-01 00:00:00',
        '15/09/2025',
        '2027-01-01',
        '31/08/2026',
    ]
})
date_source_snapshot = date_raw.copy(deep=True)
print(date_raw.to_string(index=False))

### Q 8.1. Normalize, parse and diagnose dates

Using the documented rules and `date_formats` from the worked example:

1. Strip surrounding spaces in a separate nullable string Series. Record `source_missing` before cleaning and `missing_marker` for blank text, `N/A` and `NULL` (case-insensitive).
2. Replace these markers with missing values. Parse the four accepted formats into `parsed_dates`; preserve hours, minutes and seconds.
3. Build `missing_date`, `invalid_date` (present after normalization but unparseable), and `outside_period` masks. Use September 1 inclusive and October 1 exclusive.

Explain how `03/09/2026`, `09/15/2026`, `2026-09-1O` and `2026-09-30 23:59:59` are treated. Why should the letter O not be silently replaced with zero?

> **Checkpoint:** 2 source nulls + 4 missing markers = **6 missing dates**; **7 invalid dates**; **5 outside-period dates**.

In [ ]:
# Your code and explanation here.

### Q 8.2. Retain valid rows and explain rejections

Combine the masks into `reject_date`. Create `accepted_dates` with the **original `registered_on` text** and a new **`registered_at` parsed timestamp**. Create `rejected_dates` retaining source values, parsed timestamps where available, and a `rejection_reason` (`missing_date`, `invalid_date` or `outside_period`). These reasons are mutually exclusive in this exercise.

Verify row balance and confirm that `date_raw` is unchanged. Check both September boundaries, including late-night timestamps on September 30. Display the retained and rejected rows so the source can be compared with the interpretation.

> **Checkpoint:** **18 accepted rows and 18 rejected rows**. Explain why missing markers can be normalized safely under the supplied rules, while uncertain typos cannot be repaired automatically.

In [ ]:
# Your code and explanation here.

## 9. Mini-project 1

Apply the preparation steps to a new dataset using the [Mini-project 1 brief](../prj/Project-1/BRIEF.md).

---
**1. Follow the project workflow**

> **Diagnose → normalize → decide → transform → verify → export**
>
> The service-ticket dataset has its own validity and rejection rules. Read **Required procedure** and **Acceptance checks** in the brief for dates, conflicting IDs, validity checks and the audit.

---
**2. Use the rules for each field**

> Do not copy Titanic's imputation policy into every ticket field. **Satisfaction deliberately remains missing when unavailable.**
>
> Follow the brief for deliverables, GitHub submission and the individual oral defence.

---
**Assessment**

> This individual project contributes **1.0/20 (5%)**, as specified in the [Schedule](../rsc/Schedule.md).